# Calculate unit-wise difference between the Taunton-Worcester scenario and base

Aggregate differences by stroke unit.

## Code setup

In [1]:
import os
import numpy as np
import pandas as pd

from classes.calculations import calculate_extra_bits

In [2]:
dir_data = 'data'
dir_output = 'output'
# Scenario results:
file_results_comparison = 'results_full_comparison_taunworc_base'
# Reference:
file_hospitals = 'stroke_hospitals_2025.csv'
file_unit_combos = 'unit_combos.csv'
# Output:
file_comparison_unit_combo = 'comparison_by_unit_combo_base_taunworc'

In [3]:
strings_tl = ['', '_transferloss0!67']  # column names for transfer rates
scenarios = ['base', 'taunworc']

## Load data

### Hospital reference

In [4]:
# Load hospital coordinates from CSV and create a GeoDataFrame
p = os.path.join(dir_data, file_hospitals)
df_hospitals = pd.read_csv(p, index_col='Postcode')

# Only keep hospitals in England:
df_hospitals = df_hospitals.loc[df_hospitals["Country"] == "England"]

In [5]:
display(df_hospitals.head(3))

,SSNAP name,Country,SSNAP Region,Easting,Northing,long,lat,Hospital_City,LSOA11CD,Use_IVT,Use_MT
Postcode,,,,,,,,,,,
OX39DU,John Radcliffe Hospital,England,"Buckinghamshire, Oxford and Berkshire",453940,207531,-1.219806,51.763871,Oxford,E01028534,1,1
RG15AN,Royal Berkshire Hospital,England,"Buckinghamshire, Oxford and Berkshire",472411,172962,-0.959330,51.451022,Reading,E01016414,1,0
HP112TT,Wycombe General Hospital,England,"Buckinghamshire, Oxford and Berkshire",486386,192694,-0.753428,51.626442,High Wycombe,E01017839,1,0


In [6]:
print(len(df_hospitals))

131


In [7]:
print(len(df_hospitals.index) == len(df_hospitals.index.unique()))

True


### Unit combinations


In [8]:
p = os.path.join(dir_output, file_unit_combos)
df_unit_combos = pd.read_csv(p)

In [9]:
df_unit_combos.head(3)

,unit_combo,nearest_ivt_unit_base,nearest_ivt_unit_taunworc,transfer_unit_base,transfer_unit_taunworc,standardbias,bias10min
0,42000,BN25BE,BN25BE,BN25BE,BN25BE,1,0
1,42001,BN112DH,BN112DH,BN25BE,BN25BE,1,0
2,42002,CA27HY,CA27HY,NE14LP,NE14LP,1,0


Gather SSNAP names and hospital cities too:

In [10]:
cols_unit_postcodes = ['nearest_ivt_unit', 'transfer_unit']
cols_units_here = [f'{c}_{s}' for c in cols_unit_postcodes
                   for s in scenarios]

In [11]:
cols_hosp = ['SSNAP name', 'Hospital_City']
for unit_col in cols_units_here:
    # Set up names for new columns:
    t = '_transfer_unit' if 'transfer' in unit_col else ''
    s = unit_col.split('_')[-1]  # either 'base' or 'taunworc'
    cols_new = [f'{c}{t}_{s}' for c in cols_hosp]
    rename_dict = dict(zip(cols_hosp, cols_new))
    # Merge in unit names:
    df_unit_combos = pd.merge(
        df_unit_combos,
        df_hospitals[cols_hosp].rename(columns=rename_dict).reset_index(),
        left_on=unit_col,
        right_on='Postcode',
        how='left'
    ).drop('Postcode', axis='columns')

Check results:

In [12]:
df_unit_combos.head(3)

,unit_combo,nearest_ivt_unit_base,nearest_ivt_unit_taunworc,transfer_unit_base,transfer_unit_taunworc,standardbias,bias10min,SSNAP name_base,Hospital_City_base,SSNAP name_taunworc,Hospital_City_taunworc,SSNAP name_transfer_unit_base,Hospital_City_transfer_unit_base,SSNAP name_transfer_unit_taunworc,Hospital_City_transfer_unit_taunworc
0,42000,BN25BE,BN25BE,BN25BE,BN25BE,1,0,Royal Sussex County Hospital,Brighton,Royal Sussex County Hospital,Brighton,Royal Sussex County Hospital,Brighton,Royal Sussex County Hospital,Brighton
1,42001,BN112DH,BN112DH,BN25BE,BN25BE,1,0,Worthing Hospital,Worthing,Worthing Hospital,Worthing,Royal Sussex County Hospital,Brighton,Royal Sussex County Hospital,Brighton
2,42002,CA27HY,CA27HY,NE14LP,NE14LP,1,0,Cumberland Infirmary,Carlisle,Cumberland Infirmary,Carlisle,Royal Victoria Infirmary,Newcastle,Royal Victoria Infirmary,Newcastle


In [13]:
df_unit_combos.columns

Index(['unit_combo', 'nearest_ivt_unit_base', 'nearest_ivt_unit_taunworc',
       'transfer_unit_base', 'transfer_unit_taunworc', 'standardbias',
       'bias10min', 'SSNAP name_base', 'Hospital_City_base',
       'SSNAP name_taunworc', 'Hospital_City_taunworc',
       'SSNAP name_transfer_unit_base', 'Hospital_City_transfer_unit_base',
       'SSNAP name_transfer_unit_taunworc',
       'Hospital_City_transfer_unit_taunworc'],
      dtype='str')

## Set up column names

Gather columns present in the main results dataframes.

Gather columns by type and without tailing scenario names (`_base`, `_taunworc`):

In [14]:
col_lsoa11nm = 'area_lower'
col_admissions = 'Admissions_2025_Est'
cols_mt_admissions = ['admissions_mt']
cols_transfer = ['transfers_mt']
cols_travel_time = ['drip_ship_mt_travel_time']
cols_outcome = [
    'drip_ship_mrs_0-2',
    'lvo_drip_ship_ivt_mt_mrs_0-2',
]
cols_admissions_weighted_outcomes = [
    'admissions_weighted_drip_ship_mrs_0-2',
    'admissions_weighted_lvo_drip_ship_ivt_mt_mrs_0-2',
]
cols_change = ['change_in_drip_ship_mrs_0-2',
               'change_in_lvo_drip_ship_ivt_mt_mrs_0-2']


# Add columns affected by lossy transfer rate:
def add_str_to_columns(cols, s_list):
    return [f'{c}{s}' for c in cols for s in s_list]
    
cols_mt_admissions = add_str_to_columns(cols_mt_admissions, strings_tl)
cols_transfer = add_str_to_columns(cols_transfer, strings_tl)
cols_outcome = add_str_to_columns(cols_outcome, strings_tl)
cols_admissions_weighted_outcomes = add_str_to_columns(
    cols_admissions_weighted_outcomes, strings_tl)
cols_change = add_str_to_columns(cols_change, strings_tl)

# Final gather:
cols_change += ['change_in_drip_ship_mt_travel_time', 'changed']
cols_shared = [col_lsoa11nm, col_admissions] + cols_change

## Define functions

Create some functions for the calculations.

In [15]:
def calculate_admissions_by_unit(
        df,
        cols_admissions
    ):
    """
    """
    df_admissions = (
        df[['unit_combo'] + cols_admissions].groupby('unit_combo').sum())
    
    df_admissions = df_admissions.fillna(0.0)
    df_admissions = df_admissions.round(1)
    return df_admissions

In [16]:
def calculate_outcomes_by_unit(
    df,  # by LSOA
    cols_admissions_weighted_outcomes,
    col_admissions
    ):
    """
    """
    # First sum by all patients:
    df_unit_outcomes = (
        df[['unit_combo', col_admissions] + cols_admissions_weighted_outcomes]
        .groupby('unit_combo').sum()
    )
    # Divide outcomes by the direct admissions to each unit:
    df_unit_outcomes[cols_admissions_weighted_outcomes] = (
        df_unit_outcomes[cols_admissions_weighted_outcomes]
            .div(df_unit_outcomes[col_admissions], axis='rows')
    )
    df_unit_outcomes = df_unit_outcomes.drop(col_admissions, axis='columns')
    df_unit_outcomes = df_unit_outcomes.round(5)
    return df_unit_outcomes


Use the following function to run calculations separately for each scenario, otherwise it's difficult to see what's happening.

In [17]:
def calculate_results_per_scenario(
        df,
        scenario,
        df_unit_combos,
        cols_shared,
        col_admissions,
        cols_unit_postcodes,
        cols_outcome,
    ):
    """
    """
    print(scenario)
    cols = (['unit_combo'] + cols_shared +
            [c for c in df.columns if c.endswith(scenario)])
    df_here = df[cols]
    rename_dict = dict([(c, c.replace(f'_{scenario}', ''))
                        for c in df_here.columns])
    df_here = df_here.rename(columns=rename_dict)

    # Link each first unit to its MT unit.
    # Only keep data for this scenario:
    cols_here = ['unit_combo'] + [c for c in df_unit_combos if scenario in c]
    df_unit_info = df_unit_combos[cols_here].set_index('unit_combo').copy()
    # Only keep combos in the data here:
    combos_here = df['unit_combo'].unique()
    df_unit_info = df_unit_info[df_unit_info.index.isin(combos_here)]
    # Rename columns to match data here:
    rename_dict = dict([(c, c.replace(f'_{scenario}', ''))
                        for c in df_unit_info.columns])
    df_unit_info = df_unit_info.rename(columns=rename_dict)
    print('df_unit_info:', df_unit_info.columns)
    # display(df_unit_info)
    
    # Sum admissions to each unit by scenario
    df_admissions = calculate_admissions_by_unit(
        df_here,
        [col_admissions] + cols_mt_admissions + cols_transfer
    )
    print('df_admissions:', df_admissions.columns)
    # display(df_admissions)

    # Group by unit:
    df_unit_outcomes = calculate_outcomes_by_unit(
        df_here,
        cols_admissions_weighted_outcomes,
        col_admissions
    )
    print('df_unit_outcomes:', df_unit_outcomes.columns)
    # display(df_unit_outcomes)

    # Gather all results:
    comparison_by_unit_combo = pd.concat(
        (df_unit_info, df_admissions, df_unit_outcomes),
        axis='columns'
    )
    return comparison_by_unit_combo

## Run calculations

Gather these separate functions into a main one:

In [18]:
def main(df, df_unit_combos, scenarios,
        cols_shared,
        col_admissions,
        cols_unit_postcodes,
        cols_outcome
    ):
    # Calculate unit-level data for each scenario:
    comparison_by_unit_combo_base = calculate_results_per_scenario(
        df,
        'base',
        df_unit_combos,
        cols_shared,
        col_admissions,
        cols_unit_postcodes,
        cols_outcome,
    )
    print('')
    comparison_by_unit_combo_taunworc = calculate_results_per_scenario(
        df,
        'taunworc',
        df_unit_combos,
        cols_shared,
        col_admissions,
        cols_unit_postcodes,
        cols_outcome,
    )
    
    
    # Gather and simplify info
    # Place all results in same dataframe:
    rename_base = dict([(c, f'{c}_base')
                        for c in comparison_by_unit_combo_base.columns])
    rename_taunworc = dict([(c, f'{c}_taunworc')
                           for c in comparison_by_unit_combo_taunworc.columns])
    
    comparison_by_unit_combo = pd.concat(
        (comparison_by_unit_combo_base.rename(columns=rename_base),
         comparison_by_unit_combo_taunworc.rename(columns=rename_taunworc)),
        axis='columns'
    ).sort_index()
    
    # Check that there is one entry for each combination of units:
    print('Unit combos as expected:',
          (len(comparison_by_unit_combo) ==
           len(comparison_by_unit_combo.index.unique()))
         )

    # Extra calculations for comparing scenarios:
    cols = comparison_by_unit_combo.columns.tolist()
    comparison_by_unit_combo = calculate_extra_bits(comparison_by_unit_combo,
                                                    scenarios)
    # Print new columns:
    new_cols = list(set(comparison_by_unit_combo.columns.tolist()) - set(cols))
    print('comparison_by_unit_combo:', new_cols)
    
    # Reorder columns:
    col_order = [
        'nearest_ivt_unit', 'transfer_unit', 'SSNAP name', 'Hospital_City',
        'SSNAP name_transfer_unit', 'Hospital_City_transfer_unit',
    ]
    col_order = [f'{c}_{s}' for c in col_order for s in ['base', 'taunworc']]
    col_order += [c for c in comparison_by_unit_combo.columns if c not in col_order]
    
    comparison_by_unit_combo = comparison_by_unit_combo[col_order]

    return comparison_by_unit_combo

## Run calculations

This cell prints the columns that are created by each function in calculate_results_per_scenario().

In [19]:
for s_bias in ['_bias10min', '']:
    # Load in the LSOA-level outcomes and admissions data:
    p = os.path.join(dir_output, file_results_comparison + s_bias + '.csv')
    df = pd.read_csv(p, index_col=0)
    
    # Merge unit combination labels into the main dataframe:
    df = pd.merge(df, df_unit_combos, on=cols_units_here, how='left')

    # Main calculations:
    comparison_by_unit_combo = main(
        df,
        df_unit_combos,
        scenarios,
        cols_shared,
        col_admissions,
        cols_unit_postcodes,
        cols_outcome
        )
    print('')
    # Save results with all unit combos present in df:
    p = os.path.join(dir_output, file_comparison_unit_combo + s_bias + '.csv')
    comparison_by_unit_combo.to_csv(p, index=True)

base
df_unit_info: Index(['nearest_ivt_unit', 'transfer_unit', 'SSNAP name', 'Hospital_City',
       'SSNAP name_transfer_unit', 'Hospital_City_transfer_unit'],
      dtype='str')
df_admissions: Index(['Admissions_2025_Est', 'admissions_mt',
       'admissions_mt_transferloss0!67', 'transfers_mt',
       'transfers_mt_transferloss0!67'],
      dtype='str')
df_unit_outcomes: Index(['admissions_weighted_drip_ship_mrs_0-2',
       'admissions_weighted_drip_ship_mrs_0-2_transferloss0!67',
       'admissions_weighted_lvo_drip_ship_ivt_mt_mrs_0-2',
       'admissions_weighted_lvo_drip_ship_ivt_mt_mrs_0-2_transferloss0!67'],
      dtype='str')

taunworc
df_unit_info: Index(['nearest_ivt_unit', 'transfer_unit', 'SSNAP name', 'Hospital_City',
       'SSNAP name_transfer_unit', 'Hospital_City_transfer_unit'],
      dtype='str')
df_admissions: Index(['Admissions_2025_Est', 'admissions_mt',
       'admissions_mt_transferloss0!67', 'transfers_mt',
       'transfers_mt_transferloss0!67'],
      dtyp